In [3]:
import pandas as pd

df = pd.read_csv('../data/sales_history.csv')

df.isna().sum()

df_test = df.copy()

In [4]:
df_ordenat = df_test.sort_values(by=['channel','product_line', 'region', 'month'], ascending=True)
conteo = df_ordenat.groupby(['channel','product_line', 'region'])['month'].size().reset_index(name='counts')
df_ordenat = df_ordenat.sort_values(["channel", "product_line", "region", "month"]).copy()

grup = df_ordenat.groupby(["channel", "product_line", "region"])["sales_units"]

train = df_ordenat[df_ordenat["month"].between("2022-01", "2024-12")].copy()
val = df_ordenat[df_ordenat["month"].between("2025-01", "2025-12")].copy()
test = df_ordenat[df_ordenat["month"].between("2026-01", "2026-06")].copy()

for nombre, datos in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    print(nombre, len(datos), datos["month"].min(), datos["month"].max())

grupo_media = (train.groupby(["channel", "product_line", "region"])["sales_units"].mean().rename("pred_media").reset_index())

claves = ["channel", "product_line", "region"]

# Añadimos a cada mes las ventas de esa serie 12 meses antes.
df_con_lag = df_ordenat.copy()
df_con_lag["pred_año_anterior"] = (df_con_lag.groupby(claves)["sales_units"].shift(12))

# Nos quedamos con los meses de validación: 2025.
val_comparar = df_con_lag[df_con_lag["month"].between("2025-01", "2025-12")].copy()

# A cada fila le añadimos la media de entrenamiento de su grupo.
comparacion = val_comparar.merge(grupo_media,on=claves,how="left")

comparacion[["month", *claves, "sales_units", "pred_media", "pred_año_anterior"]].head(12)

Entrenamiento 648 2022-01 2024-12
Validación 216 2025-01 2025-12
Prueba 108 2026-01 2026-06


,month,channel,product_line,region,sales_units,pred_media,pred_año_anterior
0,2025-01,ecommerce,electric,coast,60,70.222222,55.0
1,2025-02,ecommerce,electric,coast,76,70.222222,75.0
2,2025-03,ecommerce,electric,coast,57,70.222222,82.0
3,2025-04,ecommerce,electric,coast,121,70.222222,118.0
4,2025-05,ecommerce,electric,coast,83,70.222222,90.0
5,2025-06,ecommerce,electric,coast,109,70.222222,68.0
6,2025-07,ecommerce,electric,coast,102,70.222222,92.0
7,2025-08,ecommerce,electric,coast,104,70.222222,72.0
8,2025-09,ecommerce,electric,coast,81,70.222222,79.0
9,2025-10,ecommerce,electric,coast,76,70.222222,59.0
